In [ ]:
import os, awq, shutil, torch
from transformers import AutoTokenizer
from huggingface_hub import HfApi, login
from awq import AutoAWQForCausalLM

os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"

# --- 1. 定义路径和配置 ---
source_model_repo_id = "YOUR_HF_USERNAME/rank1-chainless-1.5b-lora" 
final_quantized_path = "/root/nfs/rank1_chainless_exp_awq-1.5B"

quantized_repo_id = "YOUR_HF_USERNAME/rank1-chainless-1.5b-awq" # 推荐在仓库名中注明量化方法 (awq)
hf_token = "YOUR_HF_TOKEN_HERE" 

# AWQ 量化配置
quant_config = { "w_bit": 4, "q_group_size": 128, "zero_point": True }

# --- 2. 登录 Hugging Face Hub ---
login(token=hf_token)

# --- 3. 执行 AWQ 量化 ---
print(f"\n--- 步骤 3: 从 Hub '{source_model_repo_id}' 加载模型并进行 AWQ 量化 ---")

# 加载模型以进行量化 AutoAWQ 会自动从 Hub 下载模型
awq_model = AutoAWQForCausalLM.from_pretrained(source_model_repo_id, safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(source_model_repo_id)

# 执行量化
print("🚀 开始量化")
awq_model.quantize(tokenizer, quant_config=quant_config)
print("✅ 量化完成！")

# --- 4. 保存量化后的模型 ---
if os.path.exists(final_quantized_path):
    shutil.rmtree(final_quantized_path)
# 使用 .save_quantized 方法保存
awq_model.save_quantized(final_quantized_path)
tokenizer.save_pretrained(final_quantized_path)
print(f"✅ 量化模型已保存至: {final_quantized_path}")

# --- 5. 上传到 Hugging Face Hub ---
print(f"\n--- 步骤 5: 上传量化模型到新的 Hub 仓库 ---")
api = HfApi()
api.create_repo(repo_id=quantized_repo_id, exist_ok=True)
api.upload_folder(
    folder_path=final_quantized_path, # 确保上传的是量化后的模型目录
    repo_id=quantized_repo_id,
    repo_type="model"
)
print('success')

# merge_lora_then_awq_upload.py
import os, shutil, json, sys, traceback
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import HfApi, login
from peft import PeftModel, PeftConfig
from awq import AutoAWQForCausalLM

# ========= 用户可改区 =========
LORA_REPO_ID = "YOUR_HF_USERNAME/rank1-chainless-3b-lora"            # 你的 LoRA 权重
QUANT_REPO_ID = "YOUR_HF_USERNAME/rank1-chainless-3b-awq"            # 量化后要上传的新仓库名
OUT_MERGED_DIR = "./merged-qwen2-3b"                          # 合并后本地保存目录
OUT_QUANT_DIR  = "./quantized-qwen2-3b-awq"                   # 量化后本地保存目录
QUANT_CONFIG = {"w_bit": 4, "q_group_size": 128, "zero_point": True}
# =================================

def rm_if_exists(p: str):
    if os.path.exists(p):
        shutil.rmtree(p)

def main():
    # 1) 登录
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        print("ERROR: 请先设置环境变量 HF_TOKEN，例如：export HF_TOKEN=hf_xxx", file=sys.stderr)
        sys.exit(1)
    login(token=hf_token)
    print("✅ 已登录 Hugging Face（使用环境变量 HF_TOKEN）。")

    # 2) 自动检测 LoRA 的 base model
    peft_cfg = PeftConfig.from_pretrained(LORA_REPO_ID)
    base_repo = peft_cfg.base_model_name_or_path
    print(f"🔎 检测到 LoRA 基座模型：{base_repo}")

    # 3) 合并 LoRA → 基座
    print("\n--- 合并 LoRA 到基座 ---")
    rm_if_exists(OUT_MERGED_DIR)

    # 说明：用 auto dtype + device_map=auto，这样会尽可能放到 GPU
    base = AutoModelForCausalLM.from_pretrained(base_repo, torch_dtype="auto", device_map="auto")
    lora = PeftModel.from_pretrained(base, LORA_REPO_ID)
    merged = lora.merge_and_unload()  # 把 LoRA 权重写回到 base 层
    # 有些显存吃紧的环境，合并后可以把模型先移到 CPU 再存
    merged.to("cpu")
    Path(OUT_MERGED_DIR).mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(OUT_MERGED_DIR, safe_serialization=True)

    # tokenizer：优先用基座的
    tok = AutoTokenizer.from_pretrained(base_repo, use_fast=True)
    tok.save_pretrained(OUT_MERGED_DIR)
    print(f"✅ LoRA 已合并并保存到：{OUT_MERGED_DIR}")

    # 4) AWQ 量化
    print("\n--- AWQ 量化 ---")
    rm_if_exists(OUT_QUANT_DIR)
    awq_model = AutoAWQForCausalLM.from_pretrained(OUT_MERGED_DIR, safetensors=True)
    tokenizer = AutoTokenizer.from_pretrained(OUT_MERGED_DIR, use_fast=True)

    # 量化（需要 GPU；如为多卡/小显存，可减少 max_calib_samples 或设置 n_parallel_calib_samples=1）
    awq_model.quantize(
        tokenizer,
        quant_config=QUANT_CONFIG,
        # 可选的控制项（显存紧张时启用）
        # max_calib_samples=64,
        # n_parallel_calib_samples=1,
    )
    print("✅ 量化完成。")

    # 5) 保存量化模型
    awq_model.save_quantized(OUT_QUANT_DIR)
    tokenizer.save_pretrained(OUT_QUANT_DIR)
    print(f"✅ 量化模型已保存到：{OUT_QUANT_DIR}")

    # 6) 上传到 Hugging Face
    print("\n--- 上传到 Hugging Face ---")
    api = HfApi()
    api.create_repo(repo_id=QUANT_REPO_ID, exist_ok=True, repo_type="model")
    api.upload_folder(folder_path=OUT_QUANT_DIR, repo_id=QUANT_REPO_ID, repo_type="model")
    print(f"🎉 上传完成：{QUANT_REPO_ID}")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print("❌ 发生异常：", e, file=sys.stderr)
        traceback.print_exc()
        sys.exit(2)
